# 03 Tools and MCP Server

**Owner.** Tim (Agent Orchestration & Safety Engineer) · **Supports.** Jack

**Goal.** Our knowledge base becomes a set of tools that any agent can call through the Model Context Protocol.
The agent looks articles up instead of inventing them.

**Definition of Done.** The server `fedlex-mcp` runs with `get_article`, `search_law` and `search_decisions`, and an
agent calls them on its own (task T13).

**Prerequisite.** Notebook 02 has written `data/or_articles.jsonl`.

## 1. Why MCP and not just a Python function?

A tool written as a plain function only works inside this one program. An MCP server offers the same tools to every
client over a standard protocol: our LangGraph agents, Claude Desktop, an IDE. We build the lookup once, and
experiment F2 can compare "RAG inside the prompt" with "RAG through tools" on exactly the same data.

The tool **descriptions** matter as much as the code. The model only sees name, description and parameters,
and decides from those when to call which tool.

## 2. Setup

In [3]:
import os, sys, pathlib
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()
LLM_BASE_URL = "https://litellm.engines.aisl.science/v1"
LLM_MODEL = "azure_ai/gpt-oss-120b"

llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL, api_key=os.environ["LITELLM_API_KEY"], temperature=0)
pathlib.Path("mcp_server").mkdir(exist_ok=True)
# T11: the knowledge base is data/or_articles.jsonl in the repository (GitHub integration, folder swiss_legal_agent),
# not a local copy. Pull the integration on main after a merge.
REPO_ROOT = pathlib.Path("swiss_legal_agent").resolve()
sys.path.insert(0, str(REPO_ROOT))
from lexgraph.rag.ingest import ARTICLES_FILE, load_articles
assert ARTICLES_FILE.exists(), "Pull the swiss_legal_agent integration (main) first."

Note: you may need to restart the kernel to use updated packages.
/root/venv/bin/python


## 3. The server

The cell writes the server to `mcp_server/fedlex_server.py`. In the repository it lives in `/mcp_server`.
For now `search_law` uses BM25 so the server starts without a key. In W4 it calls `retrieve()` from notebook 02.

In [5]:
%%writefile mcp_server/fedlex_server.py
"""fedlex-mcp: MCP server that exposes the Swiss Code of Obligations to our agents."""
import json
import pathlib
import re

from mcp.server.fastmcp import FastMCP
from rank_bm25 import BM25Okapi

# T11: knowledge base from the repository (GitHub integration swiss_legal_agent), not a local copy
DATA = pathlib.Path(__file__).resolve().parent.parent / "swiss_legal_agent" / "data" / "or_articles.jsonl"

mcp = FastMCP("fedlex-mcp")


def _tokenize(text):
    return re.findall(r"\w+", text.lower())


def _load():
    with open(DATA, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


ARTICLES = _load()
BY_ID = {a["id"]: a for a in ARTICLES}
BM25 = BM25Okapi([_tokenize(f"{a['title']} {a['text']}") for a in ARTICLES])


@mcp.tool()
def get_article(law: str, article: str) -> dict:
    """Return one article of Swiss federal law by its number, for example law='OR', article='335c'."""
    record = BY_ID.get(f"{law.upper()}-{article.lower()}")
    if record is None:
        return {"found": False, "message": f"Art. {article} {law} is not in the knowledge base."}
    return {"found": True, **record}


@mcp.tool()
def search_law(query: str, k: int = 4) -> list[dict]:
    """Search the knowledge base for the articles that best match a legal question (German)."""
    scores = BM25.get_scores(_tokenize(query))
    ranked = sorted(range(len(ARTICLES)), key=lambda i: scores[i], reverse=True)[:k]
    return [{"id": ARTICLES[i]["id"], "title": ARTICLES[i]["title"], "text": ARTICLES[i]["text"],
             "url": ARTICLES[i]["url"]} for i in ranked if scores[i] > 0]


@mcp.tool()
def search_decisions(query: str, k: int = 3) -> list[dict]:
    """Search published Federal Supreme Court decisions (BGE). Not connected yet: returns an empty list."""
    return []


if __name__ == "__main__":
    mcp.run()   # stdio transport

Writing mcp_server/fedlex_server.py


## 4. Connecting the agent to the server

`MultiServerMCPClient` starts the server as a subprocess (stdio) and turns its tools into LangChain tools.

In [7]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    "fedlex": {
        "command": sys.executable,
        "args": ["mcp_server/fedlex_server.py"],
        "transport": "stdio",
    }
})
tools = await client.get_tools()
for t in tools:
    print(f"{t.name}: {t.description}")

get_article: Return one article of Swiss federal law by its number, for example law='OR', article='335c'.
search_law: Search the knowledge base for the articles that best match a legal question (German).
search_decisions: Search published Federal Supreme Court decisions (BGE). Not connected yet: returns an empty list.


## 5. An agent with tools

`create_agent` builds a small loop: the model decides on a tool call, the tool answers, the model continues
until it can answer. This is the question from notebook 01, now with tools.

In [9]:
from langchain.agents import create_agent

SYSTEM_PROMPT = (
    "Du bist ein juristischer Assistent für Schweizer Arbeits- und Mietrecht. "
    "Stütze jede Aussage auf einen Artikel, den du mit den Tools abgerufen hast, und zitiere ihn als 'Art. X OR'. "
    "Findest du nichts Passendes, sag das offen. Erfinde keine Artikel und keine Bundesgerichtsentscheide."
)
agent = create_agent(model=llm, tools=tools, system_prompt=SYSTEM_PROMPT)

question_bge = (
    "Nenne zwei Bundesgerichtsentscheide (BGE mit Fundstelle) zur Kündigung zur Unzeit wegen Krankheit "
    "nach Art. 336c OR und fasse jeden in einem Satz zusammen."
)
result = await agent.ainvoke({"messages": [{"role": "user", "content": question_bge}]})
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Nenne zwei Bundesgerichtsentscheide (BGE mit Fundstelle) zur Kündigung zur Unzeit wegen Krankheit nach Art. 336c OR und fasse jeden in einem Satz zusammen.
================================== Ai Message ==================================

Ich habe in der Datenbank nach Bundesgerichtsentscheidungen (BGE) gesucht, die sich mit einer Kündigung zur Unzeit wegen Krankheit nach Art. 336c OR befassen. Da die Suche nach Entscheidungen derzeit keine Treffer liefert, kann ich Ihnen leider keine konkreten BGE‑Fundstellen nennen. 

Falls Sie weitere Informationen zu Art. 336c OR selbst benötigen, kann ich Ihnen gern die einschlägigen gesetzlichen Bestimmungen nennen.


### What just happened?

Look at the message trace. The agent called `search_law` or `get_article`, and `search_decisions` returned nothing.
A good answer now says that it has no decisions instead of inventing them. If it still invents a BGE reference,
the prompt alone is not enough. That is why the citation check in notebook 05 exists.

## Your turn

- Make the description of `search_law` worse on purpose (for example "Searches stuff"). Does the agent still pick it?
- Ask a question outside the scope, such as inheritance law. What does the agent do?
- Which tool would a lawyer need next? Write its name and description before writing any code.

## Decision log

| Date | Decision | Rejected alternative | Reason |
|---|---|---|---|
| Oct 2026 | Knowledge base exposed as an MCP server | Tools as plain Python functions | One implementation for every client, and F2 compares prompt-RAG with tool-RAG on the same data. |
| | Transport: stdio | HTTP server | |
| | BGE source for `search_decisions`: | | |

## Building block to take away

`mcp_server/fedlex_server.py` plus the four lines of `MultiServerMCPClient` from section 4.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=9b9d17b7-b4fd-473b-beb2-f7ed180fbdd3' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>